# DSDE Week 03 — Decision Tree Hyperparameter Tuning

**Objective:** improve the DecisionTreeClassifier **macro-averaged F1-score above 0.8770** using the same data and train/test split as the course notebook.

Original notebook: https://github.com/pvateekul/2110403_DSDE-CEDT_2026s1/blob/main/code/Week03_ML/1_Decision_Trees_Random_Forests_v4.ipynb

**Important:** Tune hyperparameters with cross-validation **on the training set only**; evaluate the test set after model selection. The baseline reported by the course notebook is `macro F1 = 0.8770`. The tuned score is only known after running this notebook.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix

url = "https://raw.githubusercontent.com/pvateekul/2110531_DSDE_2026s1/main/datasets/diabetes.csv"
df = pd.read_csv(url)
X = df.drop("diabetes", axis=1)
y = df["diabetes"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.30, random_state=30
)
print("Full data:", df.shape, "| Train:", X_train.shape, "| Test:", X_test.shape)
print("Test class distribution:", y_test.value_counts().sort_index().to_dict())

## 1. Original Decision Tree baseline

The instructor's code uses `criterion='entropy'`, `max_depth=3`, `min_samples_leaf=10`. The instructor's saved output shows **0.8770** macro F1. We refit here so the comparison below is generated from the same live run and split.

In [ ]:
baseline = DecisionTreeClassifier(
    criterion="entropy", max_depth=3, min_samples_leaf=10,
    random_state=30
)
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_f1 = f1_score(y_test, baseline_pred, average="macro")
print(f"Baseline test macro F1: {baseline_f1:.4f}")
print(classification_report(y_test, baseline_pred, target_names=["absent", "present"], digits=4))

## 2. Cross-validated hyperparameter search

Try `criterion` (Gini vs entropy), `max_depth` (complexity), `min_samples_leaf` and `min_samples_split` (regularization), `class_weight` (rare positive cases), and `ccp_alpha` (pruning).

`GridSearchCV(scoring='f1_macro')` will optimize exactly the metric requested. 5-fold stratified CV keeps a similar positive/negative class ratio in each fold.

In [ ]:
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 4, 5, 6, 8, None],
    "min_samples_leaf": [1, 2, 5, 10],
    "min_samples_split": [2, 5, 10],
    "class_weight": [None, "balanced"],
    "ccp_alpha": [0.0, 0.0005],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=30)
search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=30),
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=0,
)
search.fit(X_train, y_train)
print("Best 5-fold CV macro F1:", round(search.best_score_, 4))
print("Best hyperparameters:", search.best_params_)

results = pd.DataFrame(search.cv_results_)
display(results[["mean_test_score", "std_test_score", "params"]]
        .sort_values("mean_test_score", ascending=False).head(10))

## 3. Final held-out test evaluation

Only now score the selected best Decision Tree on the test set. This is the number that should be compared against the 0.8770 target.

In [ ]:
best_tree = search.best_estimator_
tuned_pred = best_tree.predict(X_test)
tuned_f1 = f1_score(y_test, tuned_pred, average="macro")

print("Best hyperparameters:", search.best_params_)
print(f"Original baseline macro F1: {baseline_f1:.4f}")
print(f"Tuned Decision Tree macro F1: {tuned_f1:.4f}")
print(f"Change: {tuned_f1 - baseline_f1:+.4f}")
print("Goal > 0.8770:", "PASS ✅" if tuned_f1 > 0.8770 else "NOT YET ❌")
print("\nTuned Decision Tree classification report:")
print(classification_report(y_test, tuned_pred, target_names=["absent", "present"], digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, tuned_pred, labels=[0, 1]))

## 4. Optional — Gradient Boosting comparison

This is for comparison only: the required score should be achieved by a **DecisionTreeClassifier**, not replaced by an ensemble.

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    random_state=30,
)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_f1 = f1_score(y_test, gb_pred, average="macro")
comparison = pd.DataFrame({
    "Model": ["Original Decision Tree", "Tuned Decision Tree", "Gradient Boosting"],
    "Test Macro F1": [baseline_f1, tuned_f1, gb_f1]
}).sort_values("Test Macro F1", ascending=False)
display(comparison)
print("Gradient Boosting classification report:")
print(classification_report(y_test, gb_pred, target_names=["absent", "present"], digits=4))

## 5. Automatic Thai-language report draft

The report below prints **actual computed values**. Copy and polish it after executing the cells. Do not submit example numbers as actual measurements.

In [ ]:
why = {
    "criterion": "เปรียบเทียบ Gini กับ Entropy สำหรับการเลือกจุดแบ่งข้อมูล",
    "max_depth": "กำหนดความลึกสูงสุดเพื่อควบคุม underfitting/overfitting",
    "min_samples_leaf": "กำหนดจำนวนข้อมูลขั้นต่ำที่แต่ละ leaf เพื่อลดการจำข้อมูลฝึกละเอียดเกินไป",
    "min_samples_split": "กำหนดจำนวนข้อมูลขั้นต่ำก่อนแบ่งโหนดใหม่",
    "class_weight": "ทดลองเพิ่มน้ำหนักคลาสที่พบน้อย เพื่อให้โมเดลสนใจคลาส present",
    "ccp_alpha": "ทดลองตัดกิ่งเพื่อลดความซับซ้อนของต้นไม้",
}
print("สรุปผลการทดลอง")
print("โมเดลเดิมให้ Macro F1-score =", f"{baseline_f1:.4f}")
print("หลังปรับแต่งได้ Macro F1-score =", f"{tuned_f1:.4f}")
print("ค่าที่ดีที่สุด:", search.best_params_)
print("พารามิเตอร์ที่ทดลองปรับ:")
for key, description in why.items():
    print(f"- {key}: {description}")
print("ผลต่างคะแนน:", f"{tuned_f1 - baseline_f1:+.4f}")
print("ผ่านเกณฑ์ > 0.8770" if tuned_f1 > 0.8770 else "ยังไม่ผ่านเกณฑ์ > 0.8770; ไม่ควรแก้ตัวเลขรายงาน")
print("หมายเหตุ: ความสัมพันธ์เชิงเหตุผลของแต่ละพารามิเตอร์วิเคราะห์ได้ แต่ผลรวมเกิดจากการปรับหลายค่าพร้อมกัน")